In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [2]:
train['TotalSpend'] = (
                        train['RoomService'].fillna(0) +
                        train['FoodCourt'].fillna(0) +
                        train['ShoppingMall'].fillna(0) +
                        train['Spa'].fillna(0) +
                        train['VRDeck'].fillna(0))

train[['Deck', 'Cabin_No','Side']] = train['Cabin'].str.split('/',expand=True)
train[['GroupID', 'MemberID']] = train['PassengerId'].str.split('_', expand=True)

group_counts = train['GroupID'].value_counts()
train["Group_Size"] = train["GroupID"].map(group_counts)

In [5]:
mask_spending = (train['CryoSleep'].isna()) & (train['TotalSpend'] > 0)
train.loc[mask_spending, 'CryoSleep'] = False

spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols:
    train.loc[(train['CryoSleep'] == True) &(train[col].isna()), col] = 0

cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
for col in cat_cols:
    if col in train.columns:
        train[col] = train[col].fillna(train[col].mode()[0])

num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    if col in train.columns:
        train[col] = train[col].fillna(train[col].median())

print("Remaining missing values:")
print(train.isna().sum().sum())

train.info()

Remaining missing values:
598
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8693 non-null   object 
 2   CryoSleep     8693 non-null   bool   
 3   Cabin         8494 non-null   object 
 4   Destination   8693 non-null   object 
 5   Age           8693 non-null   float64
 6   VIP           8693 non-null   bool   
 7   RoomService   8693 non-null   float64
 8   FoodCourt     8693 non-null   float64
 9   ShoppingMall  8693 non-null   float64
 10  Spa           8693 non-null   float64
 11  VRDeck        8693 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
 14  TotalSpend    8693 non-null   float64
 15  Deck          8693 non-null   object 
 16  Cabin_No      8494 non-null   object 
 17  Side          8693 non-null   object 
 18

In [14]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import lightgbm as lgb

features_to_drop = ['Name', 'PassengerId', 'Cabin', 'GroupID', 'MemberID']
model_df = train.drop(columns=[c for c in features_to_drop if c in train.columns], errors='ignore').copy()

model_df['Transported'] = model_df['Transported'].astype(int)

cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

le = LabelEncoder()

for col in cat_cols:
    if col in model_df.columns:

        model_df[col] = model_df[col].astype(str)
        model_df[col] = le.fit_transform(model_df[col])

if 'Cabin_No' in model_df.columns:
    model_df['Cabin_No'] = pd.to_numeric(model_df['Cabin_No'], errors='coerce').fillna(-1)

X = model_df.drop('Transported', axis=1)
y = model_df['Transported']

print("Features used: ", list(X.columns))
print(X.info())

Features used:  ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'Deck', 'Cabin_No', 'Side', 'Group_Size']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    8693 non-null   int32  
 1   CryoSleep     8693 non-null   int32  
 2   Destination   8693 non-null   int32  
 3   Age           8693 non-null   float64
 4   VIP           8693 non-null   int32  
 5   RoomService   8693 non-null   float64
 6   FoodCourt     8693 non-null   float64
 7   ShoppingMall  8693 non-null   float64
 8   Spa           8693 non-null   float64
 9   VRDeck        8693 non-null   float64
 10  TotalSpend    8693 non-null   float64
 11  Deck          8693 non-null   int32  
 12  Cabin_No      8693 non-null   float64
 13  Side          8693 non-null   int32  
 14  Group_Size

In [16]:
import warnings

warnings.filterwarnings('ignore')

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    eval_metrics='logloss',
    use_label_encoder=False,
    random_state=100
)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)

xgb_scores = cross_val_score(xgb_model, X, y, cv=kf, scoring='accuracy')

print(f"XGBoost acc per fold {xgb_scores}")
print(f'XGBoost mean acc.: {xgb_scores.mean():.4f}')

XGBoost acc per fold [0.80046003 0.80621047 0.81943646 0.80552359 0.80609896]
XGBoost mean acc.: 0.8075


In [17]:
lgbm_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    random_state=100,
    verbose=-1
)

lgbm_scores = cross_val_score(lgbm_model, X, y, cv=kf, scoring='accuracy')

print(f"LightGBM acc per fold: {lgbm_scores}")
print(f"LGBM Mean Acc. : {lgbm_scores.mean():.4f}")

LightGBM acc per fold: [0.80218516 0.81426107 0.82173663 0.81012658 0.80782509]
LGBM Mean Acc. : 0.8112


In [20]:
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier(
    estimators=[('xgb', xgb_model), ('lgbm', lgbm_model)], voting='soft')

voting_scores = cross_val_score(voting_model, X, y, cv=kf, scoring='accuracy')

print(f"Ensemble mean acc. : {voting_scores.mean():.4f}")

Ensemble mean acc. : 0.8093


In [23]:
# data for submission

test = pd.read_csv("test.csv")
submission_id = test['PassengerId'].copy()

def preprocess_data(df):
    df = df.copy()

    df[['Deck', 'Cabin_No', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df[['GroupID', 'MemberID']] = df['PassengerId'].str.split('_', expand=True)

    group_counts = df['GroupID'].value_counts()
    df["Group_Size"] = df["GroupID"].map(group_counts)

    df['TotalSpend'] = (
                        df['RoomService'].fillna(0) +
                        df['FoodCourt'].fillna(0) +
                        df['ShoppingMall'].fillna(0) +
                        df['Spa'].fillna(0) +
                        df['VRDeck'].fillna(0))

    mask_spending = (df['CryoSleep'].isna()) & (df['TotalSpend'] > 0)
    df.loc[mask_spending, 'CryoSleep'] = False

    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in spend_cols:
        df.loc[(df['CryoSleep'] == True) &(df[col].isna()), col] = 0

    cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    
    cols_drop = ['Name', 'PassengerId', 'Cabin', 'GroupID', 'MemberID']
    df = df.drop(columns=[c for c in cols_drop if c in df.columns], errors='ignore')

    cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
    le = LabelEncoder()
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)
            df[col] = le.fit_transform(df[col])

    if 'Cabin_No' in df.columns:
        df['Cabin_No'] = pd.to_numeric(df['Cabin_No'], errors='coerce').fillna(-1)

    return df

X_test = preprocess_data(test)

X_test.info()
            

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4277 entries, 0 to 4276
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    4277 non-null   int32  
 1   CryoSleep     4277 non-null   int32  
 2   Destination   4277 non-null   int32  
 3   Age           4277 non-null   float64
 4   VIP           4277 non-null   int32  
 5   RoomService   4277 non-null   float64
 6   FoodCourt     4277 non-null   float64
 7   ShoppingMall  4277 non-null   float64
 8   Spa           4277 non-null   float64
 9   VRDeck        4277 non-null   float64
 10  Deck          4277 non-null   int32  
 11  Cabin_No      4277 non-null   float64
 12  Side          4277 non-null   int32  
 13  Group_Size    4277 non-null   int64  
 14  TotalSpend    4277 non-null   float64
dtypes: float64(8), int32(6), int64(1)
memory usage: 401.1 KB


In [24]:
# training the test dataset for submission

X_test = X_test.reindex(columns=X.columns, fill_value=0)

voting_model.fit(X, y)

y_pred = voting_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': submission_id,
    'Transported': y_pred.astype(bool)
})

submission.to_csv('submission.csv', index=False)
print("Submission saved")

Submission saved
